# 04 — ASR: PhoWhisper Fine-tuning

Fine-tunes `vinai/PhoWhisper-large` on ViMD as a Vietnamese-specialized ASR model.
PhoWhisper is pre-trained on Vietnamese speech and achieves strong baselines.

**Use as**: Fallback ASR when Whisper-large-v3 is unavailable or for faster inference.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

PROJECT_ROOT = '/content/drive/MyDrive/VietSpeech'
DATASET_PATH = f'{PROJECT_ROOT}/hf_dataset'
CHECKPOINT_DIR = f'{PROJECT_ROOT}/checkpoints/phowhisper'
EXPORT_DIR = f'{PROJECT_ROOT}/weights/phowhisper'

import sys, os, torch
sys.path.insert(0, PROJECT_ROOT)
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(EXPORT_DIR, exist_ok=True)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')

In [ ]:
!pip install -q transformers datasets accelerate evaluate jiwer soundfile mlflow
print('Deps installed')

In [ ]:
from datasets import load_from_disk, Audio
from transformers import WhisperProcessor, WhisperForConditionalGeneration
from transformers import Seq2SeqTrainingArguments, Seq2SeqTrainer
import evaluate
from dataclasses import dataclass
from typing import Any, Dict, List, Union

MODEL_NAME = 'vinai/PhoWhisper-large'

ds = load_from_disk(DATASET_PATH).cast_column('audio', Audio(sampling_rate=16000))

processor = WhisperProcessor.from_pretrained(MODEL_NAME, language='Vietnamese', task='transcribe')
model = WhisperForConditionalGeneration.from_pretrained(MODEL_NAME).to(device)
model.config.forced_decoder_ids = processor.get_decoder_prompt_ids(language='vi', task='transcribe')
model.config.suppress_tokens = []

print(f'PhoWhisper-large loaded')

In [ ]:
# ── Preprocessing ─────────────────────────────────────────────────────────
def preprocess(batch):
    audio = [a['array'] for a in batch['audio']]
    features = processor.feature_extractor(audio, sampling_rate=16000, return_tensors='pt')
    batch['input_features'] = features.input_features
    labels = processor.tokenizer(
        batch['transcript'], return_tensors='pt', padding=True, truncation=True, max_length=448
    )
    batch['labels'] = labels.input_ids
    return batch

ds_proc = ds.map(preprocess, batched=True, batch_size=8, num_proc=1, remove_columns=['audio'])
ds_proc.set_format(type='torch', columns=['input_features', 'labels'])
print('Preprocessing done')

In [ ]:
# ── Data Collator + Metrics ───────────────────────────────────────────────
metric = evaluate.load('wer')

@dataclass
class DataCollatorSpeech:
    processor: Any
    def __call__(self, features):
        input_features = [{'input_features': f['input_features']} for f in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors='pt')
        label_features = [{'input_ids': f['labels']} for f in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors='pt')
        labels = labels_batch['input_ids'].masked_fill(labels_batch.attention_mask.ne(1), -100)
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]
        batch['labels'] = labels
        return batch

def compute_metrics(pred):
    pred_ids = pred.predictions
    label_ids = pred.label_ids
    label_ids[label_ids == -100] = processor.tokenizer.pad_token_id
    pred_str = processor.batch_decode(pred_ids, skip_special_tokens=True)
    label_str = processor.batch_decode(label_ids, skip_special_tokens=True)
    return {'wer': 100 * metric.compute(predictions=pred_str, references=label_str)}

data_collator = DataCollatorSpeech(processor=processor)

In [ ]:
# ── Training ──────────────────────────────────────────────────────────────
training_args = Seq2SeqTrainingArguments(
    output_dir=CHECKPOINT_DIR,
    per_device_train_batch_size=8,
    gradient_accumulation_steps=2,
    learning_rate=5e-5,
    warmup_steps=100,
    max_steps=3000,
    fp16=True,
    evaluation_strategy='steps',
    eval_steps=500,
    save_steps=500,
    load_best_model_at_end=True,
    metric_for_best_model='wer',
    greater_is_better=False,
    predict_with_generate=True,
    generation_max_length=448,
    logging_steps=25,
    report_to=['mlflow'],
    push_to_hub=False,
    remove_unused_columns=False,
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=ds_proc['train'],
    eval_dataset=ds_proc['val'],
    tokenizer=processor.feature_extractor,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

trainer.train()
trainer.save_model(EXPORT_DIR)
processor.save_pretrained(EXPORT_DIR)

print(f'PhoWhisper model saved to {EXPORT_DIR}')

In [ ]:
# ── Test evaluation ───────────────────────────────────────────────────────
test_results = trainer.predict(ds_proc['test'])
test_wer = compute_metrics(test_results)
print(f'Test WER: {test_wer["wer"]:.2f}%')
print(f'Target: ≤ 8.0% → {"PASS" if test_wer["wer"] <= 8.0 else "FAIL"}')

## ✓ PhoWhisper Fine-tuning Complete

**Model saved to**: `{EXPORT_DIR}/`

**To use locally**:
Copy `VietSpeech/weights/phowhisper/` from Google Drive to `storage/weights/phowhisper/`

**Next**: Run `05_translation_mbart.ipynb`